# 01 — Explore the Dataset

**Pipeline:** Dataset → **EDA** → Preprocessing → Augmentation → Training → Evaluation → Error Analysis → Inference → Deployment

Before touching a model, answer three questions:
1. **What classes exist, and how many images per class?** (imbalance changes everything downstream)
2. **What do the images actually look like?** (resolution, channels, artifacts)
3. **Could a human tell the classes apart?** (if not, check the labels before blaming the model)

The kit is dataset-agnostic: point `data.root` in `configs/config.yaml` at any
`root/<class>/*.jpg` folder and rerun this notebook. The example project uses the
**NEU surface-defect dataset** — six kinds of steel-surface defects (see README for the download).

In [ ]:
# run from notebooks/ -- make the repo root importable and the cwd
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

from src.utils import load_config, seed_everything
cfg = load_config("configs/config.yaml")
seed_everything(cfg["seed"])
print("project:", cfg["project_name"], "| data root:", cfg["data"]["root"])

## Class inventory

In [ ]:
from src.dataset import build_datasets
import matplotlib.pyplot as plt

ds = build_datasets(cfg)
classes = ds["classes"]
counts = [ds["class_counts"][i] for i in range(len(classes))]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(classes, counts)
ax.set_xlabel("images"); ax.set_title("images per class")
for i, c in enumerate(counts):
    ax.text(c, i, f" {c}", va="center")
plt.tight_layout(); plt.show()

print(f"{len(classes)} classes, {sum(counts)} images total")
if max(counts) > 2 * min(counts):
    print("WARNING: >2x class imbalance -- consider a weighted sampler or loss weights")

## Look at the images

One row per class, straight off disk — no transforms. This is the single highest-value cell in the notebook.

In [ ]:
from PIL import Image
import random

per_class = 6
fig, axes = plt.subplots(len(classes), per_class, figsize=(2.1 * per_class, 2.1 * len(classes)))
by_class = {i: [p for p, t in ds["samples"] if t == i] for i in range(len(classes))}
rng = random.Random(cfg["seed"])
for row, cls in enumerate(classes):
    for col, path in enumerate(rng.sample(by_class[row], per_class)):
        ax = axes[row][col] if len(classes) > 1 else axes[col]
        ax.imshow(Image.open(path).convert("RGB")); ax.axis("off")
        if col == 0:
            ax.set_ylabel(cls, rotation=0, ha="right", va="center", fontsize=10)
            ax.axis("on"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("random samples per class (untransformed)", y=1.0)
plt.tight_layout(); plt.show()

## Image geometry

Mixed sizes or aspect ratios are fine — `Resize` in the pipeline handles them — but you want to *know* before choosing `image_size`.

In [ ]:
from collections import Counter

sizes = Counter()
for path, _ in rng.sample(ds["samples"], min(500, len(ds["samples"]))):
    with Image.open(path) as im:
        sizes[im.size + (im.mode,)] += 1
for (w, h, mode), n in sizes.most_common(10):
    print(f"{w}x{h} {mode}: {n} images")
print(f"\nconfig image_size: {cfg['data']['image_size']} (everything is resized to this square)")

## What to check before moving on

- [ ] Every class folder has images, and the names make sense
- [ ] No class is drowning the others (or you have a plan for it)
- [ ] You looked at real images and the labels match what you see
- [ ] `image_size` in the config is sensible for the native resolution

Next: **[02 — Prepare the Dataset](02_prepare_dataset.ipynb)** — splits and augmentation.